# Segmentation Training Notebook

This notebook demonstrates cell segmentation using different methods:
- Blob detection (DoG)
- Cellpose (if available)
- Watershed segmentation
- Post-processing and filtering

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage
import logging

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from biohub_tracking.data.zarr_loader import iter_frames
from biohub_tracking.segmentation.segmenter import CellSegmenter
from biohub_tracking.segmentation.postprocess import PostProcessor

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.rcParams['figure.figsize'] = (14, 6)
np.set_printoptions(precision=3, suppress=True)

## 1. Load Sample Data

In [ ]:
# Define paths
data_dir = Path("../data")
train_dir = data_dir / "train"
test_dir = data_dir / "test"

# Find a sample
sample_path = None
if train_dir.exists():
    samples = sorted(train_dir.glob("*.zarr"))
    if samples:
        sample_path = samples[0]
        print(f"Using training sample: {sample_path.name}")
elif test_dir.exists():
    samples = sorted(test_dir.glob("*.zarr"))
    if samples:
        sample_path = samples[0]
        print(f"Using test sample: {sample_path.name}")

if sample_path:
    # Load first frame
    first_frame = None
    for frame_index, image in iter_frames(sample_path):
        first_frame = image
        print(f"Frame shape: {image.shape}")
        print(f"Intensity range: [{image.min():.1f}, {image.max():.1f}]")
        break
else:
    print("No sample data found")
    first_frame = None

## 2. Blob Detection (DoG) Segmentation

In [ ]:
if first_frame is not None:
    # Initialize blob segmenter
    VOXEL_SIZE_UM = (1.625, 0.40625, 0.40625)
    blob_segmenter = CellSegmenter(
        method="blob",
        min_size=30,
        anisotropy=VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1],
        voxel_size_um=VOXEL_SIZE_UM
    )
    
    print("Segmenting with Blob Detection...")
    labels, cells = blob_segmenter.segment_frame(first_frame, frame_index=0)
    
    print(f"Number of cells detected: {len(cells)}")
    print(f"Label mask shape: {labels.shape}")
    print(f"Unique labels: {np.unique(labels).size}")
    
    if cells:
        print(f"\nSample cell properties:")
        for i, cell in enumerate(cells[:3]):
            print(f"  Cell {i}: {cell}")

## 3. Visualize Segmentation Results

In [ ]:
if first_frame is not None and labels is not None:
    # Get middle Z slice
    z_middle = first_frame.shape[0] // 2
    img_slice = first_frame[z_middle, :, :]
    label_slice = labels[z_middle, :, :]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Original image
    im0 = axes[0].imshow(img_slice, cmap='gray')
    axes[0].set_title(f'Original Frame (Z={z_middle})')
    axes[0].set_xlabel('X pixels')
    axes[0].set_ylabel('Y pixels')
    plt.colorbar(im0, ax=axes[0])
    
    # Segmentation labels
    im1 = axes[1].imshow(label_slice, cmap='nipy_spectral')
    axes[1].set_title(f'Segmented Labels (Z={z_middle})')
    axes[1].set_xlabel('X pixels')
    axes[1].set_ylabel('Y pixels')
    plt.colorbar(im1, ax=axes[1])
    
    # Overlay
    axes[2].imshow(img_slice, cmap='gray')
    axes[2].imshow(label_slice, cmap='nipy_spectral', alpha=0.4)
    axes[2].set_title('Overlay: Image + Segmentation')
    axes[2].set_xlabel('X pixels')
    axes[2].set_ylabel('Y pixels')
    
    plt.tight_layout()
    plt.show()
    
    # Plot cell size distribution
    if cells:
        sizes = [cell.get('volume', cell.get('size', 0)) for cell in cells]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(sizes, bins=30, edgecolor='black', alpha=0.7)
        ax.set_xlabel('Cell Volume (voxels)')
        ax.set_ylabel('Frequency')
        ax.set_title('Distribution of Cell Sizes')
        ax.axvline(np.mean(sizes), color='r', linestyle='--', label=f'Mean: {np.mean(sizes):.1f}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.show()

## 4. Segment Multiple Frames

In [ ]:
if sample_path:
    print(f"Segmenting all frames from {sample_path.name}...")
    all_segmentations = {}
    all_cells = {}
    
    frame_count = 0
    max_frames = 10  # Limit to first 10 frames for speed
    
    for frame_index, image in iter_frames(sample_path):
        if frame_count >= max_frames:
            break
            
        labels, cells = blob_segmenter.segment_frame(image, frame_index=frame_index)
        all_segmentations[frame_index] = labels
        all_cells[frame_index] = cells
        
        print(f"Frame {frame_index}: {len(cells)} cells detected")
        frame_count += 1
    
    print(f"\nSegmentation complete. Processed {frame_count} frames.")
    
    # Summary statistics
    cell_counts = [len(cells) for cells in all_cells.values()]
    print(f"Cell count per frame: min={np.min(cell_counts)}, max={np.max(cell_counts)}, mean={np.mean(cell_counts):.1f}")

## 5. Post-Processing Options

In [ ]:
# Post-processing configuration
postprocess_config = {
    'min_volume': 30,       # Remove cells smaller than this
    'max_volume': 50000,    # Remove cells larger than this
    'remove_border': False, # Remove cells touching image border
    'smooth_labels': False  # Apply morphological smoothing
}

print("Post-processing Configuration:")
for key, value in postprocess_config.items():
    print(f"  {key}: {value}")

# Initialize post-processor
post_processor = PostProcessor(**postprocess_config)
print("\nPost-processor initialized.")

## 6. Summary

In [ ]:
print("Segmentation Training Summary:")
print("="*50)
print(f"Method: Blob Detection (DoG)")
print(f"Min size filter: 30 voxels")
print(f"Voxel size: {VOXEL_SIZE_UM} µm (Z, Y, X)")
print(f"Anisotropy ratio (Z/XY): {VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1]:.2f}")
print()
print("Key findings:")
if first_frame is not None:
    print(f"  - Input image shape: {first_frame.shape}")
    print(f"  - Intensity range: [{first_frame.min():.1f}, {first_frame.max():.1f}]")
if 'all_cells' in locals() and all_cells:
    print(f"  - Total frames processed: {len(all_cells)}")
    print(f"  - Average cells per frame: {np.mean([len(c) for c in all_cells.values()]):.1f}")
print("\nNext step: Use segmentations for tracking")